In [1]:
!nvidia-smi
import torch
torch.cuda.is_available(), torch.cuda.get_device_name(0)


Mon Sep 15 06:40:33 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   49C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

(True, 'Tesla T4')

In [2]:
!pip -q install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip -q install transformers pillow tqdm kaggle


In [3]:
from google.colab import drive
drive.mount('/content/drive')
RAW_DIR = "/content/drive/MyDrive/fashion_raw"  # <- change if needed


Mounted at /content/drive


In [4]:
import json, os, pathlib, zipfile
from google.colab import files

# Upload kaggle.json (a file picker will open)
uploaded = files.upload()  # select kaggle.json

# Install the token
os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/kaggle.json', 'wb') as f:
    f.write(uploaded['kaggle.json'])
os.chmod('/root/.kaggle/kaggle.json', 0o600)

# Example: download a fashion dataset (change to what you like on Kaggle)
!kaggle datasets list -s deepfashion | head -n 20

# Example placeholder (pick a concrete dataset you want):
# !kaggle datasets download -d <owner>/<dataset-slug> -p /content/data/raw -q
# !unzip -q -o "/content/data/raw/*.zip" -d /content/data/raw

RAW_DIR = "/content/data/raw"  # where images ended up after unzip


Saving kaggle.json to kaggle.json
ref                                                         title                                                    size  lastUpdated                 downloadCount  voteCount  usabilityRating  
----------------------------------------------------------  ------------------------------------------------  -----------  --------------------------  -------------  ---------  ---------------  
vishalbsadanand/deepfashion-1                               Deepfashion 1                                      6120367791  2023-02-15 15:04:05.520000           2429         23  0.75             
achariso/deepfashion-fisb                                   DeepFashion FISB                                  12493491809  2021-11-23 17:44:15.710000            475          2  0.9375           
hserdaraltan/deepfashion-inshop-clothes-retrieval-adjusted  DeepFashion_In-shop_Clothes_Retrieval_Adjusted     2239158848  2022-06-04 15:14:06.813000            442         18  0.875    

In [5]:
DATASET = "abdokhaled1212/deepfashion-merged-dataset-512x512"
DOWNLOAD_DIR = "/content/data/raw"

!mkdir -p "$DOWNLOAD_DIR"
!kaggle datasets download -d "$DATASET" -p "$DOWNLOAD_DIR" -q
!unzip -q -o "$DOWNLOAD_DIR"/*.zip -d "$DOWNLOAD_DIR"
!find "$DOWNLOAD_DIR" -type f | head -n 20  # peek files


Dataset URL: https://www.kaggle.com/datasets/abdokhaled1212/deepfashion-merged-dataset-512x512
License(s): CC0-1.0
/content/data/raw/deepfashion-merged-dataset-512x512.zip
/content/data/raw/photos/MEN-Tees_Tanks-id_00003434-01_3_back.jpg
/content/data/raw/photos/WOMEN-Jackets_Coats-id_00001073-01_1_front.jpg
/content/data/raw/photos/WOMEN-Dresses-id_00001552-02_1_front.jpg
/content/data/raw/photos/MEN-Sweaters-id_00001509-04_3_back.jpg
/content/data/raw/photos/WOMEN-Tees_Tanks-id_00001025-02_1_front.jpg
/content/data/raw/photos/WOMEN-Sweaters-id_00004860-06_1_front.jpg
/content/data/raw/photos/WOMEN-Blouses_Shirts-id_00007634-02_1_front.jpg
/content/data/raw/photos/WOMEN-Tees_Tanks-id_00006960-04_1_front.jpg
/content/data/raw/photos/WOMEN-Dresses-id_00005768-03_1_front.jpg
/content/data/raw/photos/MEN-Tees_Tanks-id_00004674-05_3_back.jpg
/content/data/raw/photos/WOMEN-Graphic_Tees-id_00002486-01_1_front.jpg
/content/data/raw/photos/WOMEN-Tees_Tanks-id_00001466-01_1_front.jpg
/content/d

In [6]:
# create the train/val class folders
import os, pathlib

CLASSES = ["casual","formal","sporty","vintage","modern",
           "elegant","streetwear","bohemian","minimalist","trendy"]

for split in ["train","val"]:
    for c in CLASSES:
        os.makedirs(f"/content/data/{split}/{c}", exist_ok=True)

!find /content/data -type d | sort


/content/data
/content/data/raw
/content/data/raw/photos
/content/data/train
/content/data/train/bohemian
/content/data/train/casual
/content/data/train/elegant
/content/data/train/formal
/content/data/train/minimalist
/content/data/train/modern
/content/data/train/sporty
/content/data/train/streetwear
/content/data/train/trendy
/content/data/train/vintage
/content/data/val
/content/data/val/bohemian
/content/data/val/casual
/content/data/val/elegant
/content/data/val/formal
/content/data/val/minimalist
/content/data/val/modern
/content/data/val/sporty
/content/data/val/streetwear
/content/data/val/trendy
/content/data/val/vintage


In [7]:
# Auto label images
import os, csv, random, shutil, math, gc
from pathlib import Path
from typing import List, Tuple
from PIL import Image
import torch
from transformers import CLIPProcessor, CLIPModel

STYLES = ["casual","formal","sporty","vintage","modern",
          "elegant","streetwear","bohemian","minimalist","trendy"]

RAW = Path(RAW_DIR)
OUT = Path("/content/data")
VAL_FRACTION = 0.15
SEED = 42

def is_image(p: Path) -> bool:
    return p.suffix.lower() in {".jpg",".jpeg",".png",".webp",".bmp"}

device = "cuda" if torch.cuda.is_available() else "cpu"
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
proc  = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

prompts = [f"a {s} outfit street fashion photo" for s in STYLES]

paths = [p for p in RAW.rglob("*") if is_image(p)]
print(f"Found {len(paths)} images under {RAW}")
random.seed(SEED)

assignments = []
for p in paths:
    try:
        img = Image.open(p).convert("RGB")
    except Exception:
        continue
    inputs = proc(text=prompts, images=img, return_tensors="pt", padding=True).to(device)
    with torch.no_grad():
        out = model(**inputs)
        probs = out.logits_per_image.softmax(dim=-1).squeeze().tolist()  # probs over prompts
    best_idx = int(max(range(len(probs)), key=lambda i: probs[i]))
    top_style = STYLES[best_idx]
    split = "val" if random.random() < VAL_FRACTION else "train"
    dest = OUT / split / top_style / p.name
    i = 1
    while dest.exists():
        dest = dest.with_stem(dest.stem + f"_{i}")
        i += 1
    shutil.copy2(p, dest)
    assignments.append([str(p), split, top_style, probs[best_idx]])

# Save a CSV log
OUT.mkdir(parents=True, exist_ok=True)
with open(OUT / "auto_labels.csv", "w", newline="") as f:
    wr = csv.writer(f)
    wr.writerow(["source_path","split","assigned_style","assigned_prob"])
    wr.writerows(assignments)

print("Done auto-labeling. Inspect /content/data/auto_labels.csv and sample folders.")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

Found 9800 images under /content/data/raw
Done auto-labeling. Inspect /content/data/auto_labels.csv and sample folders.


In [10]:
import os, random, shutil
from pathlib import Path
import pandas as pd

random.seed(42)

ROOT = Path("/content/data")
MIN_COUNT = 50          # drop classes below this
MAX_PER_CLASS = 1200    # cap to avoid dominance (adjust if you want)

def count_split(split):
    rows = []
    for cdir in sorted((ROOT / split).iterdir()):
        if cdir.is_dir():
            rows.append((cdir.name, len(list(cdir.glob("*")))))
    return pd.DataFrame(rows, columns=["class", split]).set_index("class")

train_counts = count_split("train")
val_counts   = count_split("val")
counts = train_counts.join(val_counts, how="outer").fillna(0).astype(int)
display(counts)

# 1) Drop tiny classes
drop = [cls for cls, row in counts.iterrows() if row["train"] + row["val"] < MIN_COUNT]
print("Dropping classes:", drop)
for split in ["train","val"]:
    for cls in drop:
        d = ROOT / split / cls
        if d.exists():
            shutil.rmtree(d)

# 2) Cap huge classes (randomly move extras to overflow so they’re ignored)
OVERFLOW = ROOT / "overflow"
OVERFLOW.mkdir(exist_ok=True)
for split in ["train","val"]:
    for cdir in sorted((ROOT / split).iterdir()):
        if not cdir.is_dir(): continue
        files = list(cdir.glob("*"))
        cap = MAX_PER_CLASS if split == "train" else int(MAX_PER_CLASS*0.2)
        if len(files) > cap:
            keep = set(random.sample(files, cap))
            for f in files:
                if f not in keep:
                    dest = OVERFLOW / f.name
                    i=1
                    while dest.exists():
                        dest = dest.with_stem(dest.stem+f"_{i}"); i+=1
                    shutil.move(str(f), str(dest))

# Recount
train_counts = count_split("train")
val_counts   = count_split("val")
counts = train_counts.join(val_counts, how="outer").fillna(0).astype(int)
display(counts)

# 3) Build f


,train,val
class,,
bohemian,1161,209
casual,190,22
elegant,251,38
formal,103,19
minimalist,1706,315
modern,2,0
sporty,1319,224
streetwear,2853,549
trendy,723,113


Dropping classes: ['modern', 'vintage']


,train,val
class,,
bohemian,1161,209
casual,190,22
elegant,251,38
formal,103,19
minimalist,1200,240
sporty,1200,224
streetwear,1200,240
trendy,723,113


In [11]:
# 4) (Optional) See class counts
import os, pandas as pd
rows = []
for split in ["train","val"]:
    for c in CLASSES:
        cnt = len(os.listdir(f"/content/data/{split}/{c}"))
        rows.append([split, c, cnt])
pd.DataFrame(rows, columns=["split","class","count"]).pivot(index="class", columns="split", values="count").fillna(0)


FileNotFoundError: [Errno 2] No such file or directory: '/content/data/train/vintage'

In [12]:
from pathlib import Path

# Use relabeled set if it exists, otherwise the first one
DATA = Path("/content/data_relabel") if (Path("/content/data_relabel/train").exists()) else Path("/content/data")
assert (DATA/"train").exists(), f"Expected {DATA}/train to exist"

# Build CLASSES dynamically from your folders (after any pruning/capping)
CLASSES = sorted([d.name for d in (DATA/"train").iterdir() if d.is_dir()])
CLASSES


['bohemian',
 'casual',
 'elegant',
 'formal',
 'minimalist',
 'sporty',
 'streetwear',
 'trendy']

In [13]:
# Setup
import os, random, numpy as np, torch, torch.nn as nn, torch.optim as optim
from pathlib import Path
from PIL import Image
from tqdm import tqdm
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
from torchvision.models import resnet18, ResNet18_Weights

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
OUT = Path("/content/models"); OUT.mkdir(exist_ok=True, parents=True)

print("Device:", device)
print("Data root:", DATA)
print("Classes:", CLASSES)


Device: cuda
Data root: /content/data
Classes: ['bohemian', 'casual', 'elegant', 'formal', 'minimalist', 'sporty', 'streetwear', 'trendy']


In [15]:
# 2) Dataloaders (+ class weights for imbalance)
from collections import Counter

train_tfms = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.7,1.0), ratio=(0.75,1.33)),
    transforms.RandomHorizontalFlip(0.5),
    transforms.ColorJitter(0.15,0.15,0.10,0.02),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])
val_tfms = transforms.Compose([
    transforms.Resize(256), transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

train_ds = ImageFolder(DATA/"train", transform=train_tfms)
val_ds   = ImageFolder(DATA/"val",   transform=val_tfms)

# Remap to our fixed class order (from disk)
idx_map = {cls:i for i,cls in enumerate(CLASSES)}

def remap(ds: ImageFolder):
    samples = []
    for path, _ in ds.samples:
        cls = Path(path).parent.name
        if cls not in idx_map:  # skip orphaned dirs if any
            continue
        samples.append((path, idx_map[cls]))
    ds.samples = samples
    ds.targets = [y for _, y in samples]
    ds.class_to_idx = idx_map

remap(train_ds); remap(val_ds)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

# Class weights (helps when counts are uneven)
counts = np.zeros(len(CLASSES), dtype=np.int64)
for _, y in train_ds.samples: counts[y]+=1
weights = counts.sum() / (len(CLASSES) * np.maximum(counts, 1))
class_weights = torch.tensor(weights, dtype=torch.float32, device=device)

len(train_ds), len(val_ds), dict(zip(CLASSES, counts.tolist()))


(6028,
 1105,
 {'bohemian': 1161,
  'casual': 190,
  'elegant': 251,
  'formal': 103,
  'minimalist': 1200,
  'sporty': 1200,
  'streetwear': 1200,
  'trendy': 723})

In [16]:
# Build the model
model = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
in_f = model.fc.in_features
model.fc = nn.Linear(in_f, len(CLASSES))
model = model.to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:01<00:00, 33.9MB/s]


In [17]:
# Eval Helper
@torch.no_grad()
def evaluate():
    model.eval(); total=0; correct=0; loss_sum=0.0
    for xb,yb in val_loader:
        xb,yb = xb.to(device), yb.to(device)
        logits = model(xb)
        loss_sum += criterion(logits, yb).item()
        pred = logits.argmax(1); correct += (pred==yb).sum().item(); total += yb.size(0)
    return (loss_sum/total if total else 0.0), (correct/total if total else 0.0)


In [18]:
# Training the head only
for p in model.parameters(): p.requires_grad=False
for p in model.fc.parameters(): p.requires_grad=True

opt = optim.AdamW(model.fc.parameters(), lr=1e-3, weight_decay=1e-4)

best = {"acc":0.0, "state":None}
E1 = 5
for epoch in range(1, E1+1):
    model.train()
    pbar = tqdm(train_loader, desc=f"Head {epoch}/{E1}")
    for xb,yb in pbar:
        xb,yb = xb.to(device), yb.to(device)
        opt.zero_grad(set_to_none=True)
        loss = criterion(model(xb), yb)
        loss.backward(); opt.step()
    vl, va = evaluate()
    print(f"[Head] val_loss={vl:.3f}  val_acc={va:.3%}")
    if va > best["acc"]:
        best = {"acc":va, "state":{k:v.cpu() for k,v in model.state_dict().items()}}

if best["state"] is not None:
    model.load_state_dict(best["state"])


Head 1/5: 100%|██████████| 189/189 [00:53<00:00,  3.56it/s]


[Head] val_loss=0.046  val_acc=45.701%


Head 2/5: 100%|██████████| 189/189 [00:54<00:00,  3.47it/s]


[Head] val_loss=0.045  val_acc=43.258%


Head 3/5: 100%|██████████| 189/189 [00:56<00:00,  3.33it/s]


[Head] val_loss=0.043  val_acc=48.054%


Head 4/5: 100%|██████████| 189/189 [00:53<00:00,  3.56it/s]


[Head] val_loss=0.039  val_acc=54.027%


Head 5/5: 100%|██████████| 189/189 [00:54<00:00,  3.46it/s]


[Head] val_loss=0.039  val_acc=53.122%


In [19]:
# Fine tune last block and head
for p in model.parameters(): p.requires_grad=False
for p in model.layer4.parameters(): p.requires_grad=True
for p in model.fc.parameters(): p.requires_grad=True

opt = optim.AdamW(filter(lambda p:p.requires_grad, model.parameters()), lr=3e-4, weight_decay=1e-4)
sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=8)

best2 = {"acc":0.0, "state":None}
E2 = 8
for epoch in range(1, E2+1):
    model.train()
    pbar = tqdm(train_loader, desc=f"FT {epoch}/{E2}")
    for xb,yb in pbar:
        xb,yb = xb.to(device), yb.to(device)
        opt.zero_grad(set_to_none=True)
        loss = criterion(model(xb), yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(filter(lambda p:p.requires_grad, model.parameters()), 5.0)
        opt.step()
    sched.step()
    vl, va = evaluate()
    print(f"[FT]  val_loss={vl:.3f}  val_acc={va:.3%}")
    if va > best2["acc"]:
        best2 = {"acc":va, "state":{k:v.cpu() for k,v in model.state_dict().items()}}

if best2["state"] is not None:
    model.load_state_dict(best2["state"])


FT 1/8: 100%|██████████| 189/189 [01:01<00:00,  3.08it/s]


[FT]  val_loss=0.038  val_acc=54.842%


FT 2/8: 100%|██████████| 189/189 [00:58<00:00,  3.21it/s]


[FT]  val_loss=0.040  val_acc=55.475%


FT 3/8: 100%|██████████| 189/189 [00:52<00:00,  3.59it/s]


...

[FT]  val_loss=0.032  val_acc=64.434%


FT 7/8: 100%|██████████| 189/189 [00:51<00:00,  3.70it/s]


[FT]  val_loss=0.032  val_acc=65.882%


FT 8/8: 100%|██████████| 189/189 [00:52<00:00,  3.58it/s]


[FT]  val_loss=0.032  val_acc=67.602%


In [20]:
# (Optional) Temperature scaling (confidence calibration)
@torch.no_grad()
def collect_logits_labels(loader):
    model.eval(); L=[]; Y=[]
    for xb,yb in loader:
        xb,yb = xb.to(device), yb.to(device)
        L.append(model(xb).cpu()); Y.append(yb.cpu())
    return torch.cat(L), torch.cat(Y)

@torch.no_grad()
def nll_with_T(logits, labels, T):
    scaled = logits / T.clamp(min=1e-4)
    logp = torch.log_softmax(scaled, dim=1)
    return nn.NLLLoss()(logp, labels)

temperature = 1.0
try:
    logits, labels = collect_logits_labels(val_loader)
    T = torch.ones(1, requires_grad=True)
    optT = optim.LBFGS([T], lr=0.5, max_iter=50, line_search_fn="strong_wolfe")
    def closure():
        optT.zero_grad()
        loss = nll_with_T(logits, labels, T)
        loss.backward(); return loss
    optT.step(closure)
    temperature = float(T.detach().item())
    print("Calibrated temperature:", temperature)
except Exception as e:
    print("Temp scaling skipped:", e)


Temp scaling skipped: element 0 of tensors does not require grad and does not have a grad_fn


In [21]:
# Saving checkpoint for app
ckpt = {
    "state_dict": {k:v.cpu() for k,v in model.state_dict().items()},
    "classes": CLASSES,
    "temperature": float(temperature),
}
OUT.mkdir(parents=True, exist_ok=True)
torch.save(ckpt, OUT/"fashion_head.pt")
print("✅ Saved:", OUT/"fashion_head.pt")


✅ Saved: /content/models/fashion_head.pt


In [22]:
# Sanity Check
from PIL import Image
from torchvision import transforms
import numpy as np, torch

# pick any val image
test_img = next((DATA/"val").rglob("*.jpg"))
print("Testing:", test_img)

prep = transforms.Compose([
    transforms.Resize(256), transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

img = prep(Image.open(test_img).convert("RGB")).unsqueeze(0).to(device)
model.eval()
with torch.no_grad():
    logits = model(img) / temperature
    probs = torch.softmax(logits, dim=1)[0].cpu().numpy()
pred = int(np.argmax(probs))
print("Pred:", CLASSES[pred], "Conf:", float(probs[pred]))


Testing: /content/data/val/sporty/WOMEN-Tees_Tanks-id_00000512-08_1_front.jpg
Pred: minimalist Conf: 0.9216419458389282
